In [8]:
%%capture
%run ../part_1/part1.ipynb
import math
from collections import Counter
from array import array
from collections import defaultdict

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/u251664/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
#We create a dictionary called field_terms for organizing the searchable terms separately for each field.
field_terms = {
    "title":       list(df["title_terms"]),
    "description": list(df["description_terms"]),
    "category":    list(df["category_terms"]),     
    "brand":       list(df["brand_terms"]),
    "seller":      list(df["seller_terms"]),
    "details":     [build_terms(details_to_text(x)) for x in df["product_details"]],   # values only
}

pids = list(df["pid"])
N = len(df)

Part 1: Indexing 

In [18]:
from collections import defaultdict
def create_index(all_terms):
    """
    Implement the inverted index of one field

    Argument:
    all_terms - list of words of all the term of all documents in that specific field

    Returns:
    index - the inverted index (implemented through a Python dictionary) containing terms as keys and the corresponding
    list of documents where these keys appears in (and the positions) as values.
    """
    index = defaultdict(list)

    for doc_id, terms in enumerate(all_terms):
        current_doc_index = {}  # Create temporary index for the current document

        for position, term in enumerate(terms):
            try:
                # if the term is already in the index for the current page (current_page_index)
                # append the position to the corresponding list (elemento 1 del arreglo, el 0 es la id del documento)
                current_doc_index[term][1].append(position)
            except:
                # Add the new term as dict key and initialize the array of positions and add the position
                current_doc_index[term] = [doc_id, array('I', [position])] #'I' indicates unsigned int (int in Python)

        #merge the current page index with the main index
        for term, posting in current_doc_index.items():
            index[term].append(posting)

    return index

In [19]:
# one inverted index per fiel
index = {}
for field in field_terms:
    index[field] = create_index(field_terms[field])
    print(field, ":", len(index[field]), "terms")

title : 738 terms
description : 4698 terms
category : 40 terms
brand : 402 terms
seller : 687 terms
details : 17812 terms


In [ ]:
#We have decided the important of these fileds based on the result of part 1 validate labels and the result of the index
weights = {"title": 3.0,"description": 1.5,"category": 0.5,"brand": 1.0, "seller": 0.0, "details": 2.0}

In [22]:
def search(query, top_k=20):
    """
    The output is the list of documents that contain any of the query terms.
    So, we will get the list of documents for each query term, and take the union of them.
    """
    #Normalize the query, so it matches with the pre-processed documents and fields
    query_terms = list(set(build_terms(query)))
    if len(query_terms) == 0:
        return []

    #Find which documents contain each term

    tf = {} #frequency 
    term_docs = {}   # pid of the docuemtns that contain the word                   
    for term in query_terms:
        term_docs[term] = set()
        for field in index:
            tf[(field, term)] = {} #empty dictionary to store the frequency of the current word in the current field
            try:
                postings = index[field][term] # Get the postings for this term in this field
                for posting in postings:
                    doc_id = posting[0]
                    positions = posting[1]
                    tf[(field, term)][doc_id] = len(positions)
                    term_docs[term].add(doc_id)
            except KeyError: # This term does not exist in this field
                pass

        if not term_docs[term]:
            return []
        

    # AND, calculates the interseccion
    docs = None
    for term in query_terms:
        if len(term_docs[term]) == 0:
            return []
        if docs is None:
            docs = term_docs[term]
        else:
            docs = docs.intersection(term_docs[term])

    # TF-IDF score
    results = []
    for doc_id in docs:
        score = 0
        for term in query_terms:
            idf = math.log(N / len(term_docs[term]))
            for field in index:
                if doc_id in tf[(field, term)]:
                    score += weights[field] * (1 + math.log(tf[(field, term)][doc_id])) * idf
        results.append((doc_id, score))

    results.sort(key=lambda x: x[1], reverse=True)
    return results[:top_k]

In [23]:
def show(query, top_k=10):
    """Print the top_k results of a query."""
    for doc_id, score in search(query, top_k):
        print(round(score, 3), pids[doc_id], df["title"].iat[doc_id], "|", df["brand"].iat[doc_id])

show("women full sleeve sweatshirt cotton")

42.869 SWSFFVK89VKGHBZN Full Sleeve Graphic Print Women Sweatshirt | CupidSto
42.869 SWSFFVK8RYHBBFAH Full Sleeve Graphic Print Women Sweatshirt | CupidSto
42.665 SWSFFVKBCQG5FHPF Full Sleeve Printed Women Sweatshirt | CupidSto
42.665 SWSFFVKBH5YEGFFN Full Sleeve Printed Women Sweatshirt | CupidSto
42.665 SWSFFVKBB8FHEB7R Full Sleeve Printed Women Sweatshirt | CupidSto
42.665 SWSFFVKBCY7RSGYJ Full Sleeve Printed Women Sweatshirt | CupidSto
42.665 SWSFFVKBQUWH72KH Full Sleeve Printed Women Sweatshirt | CupidSto
42.665 SWSFFVKBYG9ZUBTW Full Sleeve Printed Women Sweatshirt | CupidSto
42.665 SWSFFVKBF5U5AHBR Full Sleeve Printed Women Sweatshirt | CupidSto
41.228 SWSFWEF3XGHFBCJC Full Sleeve Graphic Print Women Sweatshirt | Free Authori


In [ ]:
test_queries = {
    1: "men round neck cotton shirt",
    2: "women printed topwear",
    3: "men slim fit jeans",
    4: "women cotton dress",
    5: "men solid polo shirt",
}

for qid, q in test_queries.items():
    print(qid, q, "->", len(search(q, top_k=100000)), "documents")

1 men round neck cotton shirt -> 3227 documents
2 women printed sweatshirt -> 371 documents
3 men slim fit jeans -> 212 documents
4 women cotton dress -> 517 documents
5 men solid polo shirt -> 919 documents
